# Credit Risk Prediction and Expected Loss Analysis

This notebook builds a credit default risk prediction model using borrower-level loan data.

The workflow includes:

1. Data loading and cleaning  
2. Exploratory data analysis  
3. Feature engineering  
4. Logistic Regression baseline model  
5. Random Forest model  
6. Risk band segmentation  
7. Expected loss analysis  
8. Approval strategy simulation  

**Business objective:** Estimate borrower probability of default and translate model outputs into practical credit risk decisions.


## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# Output folders
fig_dir = "../outputs/figures"
table_dir = "../outputs/tables"

os.makedirs(fig_dir, exist_ok=True)
os.makedirs(table_dir, exist_ok=True)

print("Libraries imported successfully.")

## 2. Load Data

The raw dataset should be placed in:

```text
credit-risk-prediction/data/credit_risk_dataset.csv
```


In [ ]:
file_path = "../data/credit_risk_dataset.csv"

df = pd.read_csv(file_path)

print("Data loaded successfully.")
print("Shape:", df.shape)

df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df["loan_status"].value_counts(normalize=True)

## 3. Data Cleaning

In [ ]:
df_clean = df.copy()

print("Original shape:", df_clean.shape)

# Remove unrealistic age observations
df_clean = df_clean[df_clean["person_age"] <= 100]

# Remove unrealistic employment length observations
df_clean = df_clean[df_clean["person_emp_length"].isna() | (df_clean["person_emp_length"] <= 80)]

# Fill missing employment length and interest rate with median
df_clean["person_emp_length"] = df_clean["person_emp_length"].fillna(
    df_clean["person_emp_length"].median()
)

df_clean["loan_int_rate"] = df_clean["loan_int_rate"].fillna(
    df_clean["loan_int_rate"].median()
)

# Remove duplicates
duplicates = df_clean.duplicated().sum()
print("Duplicate rows:", duplicates)

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print("Cleaned shape:", df_clean.shape)
print("Missing values after cleaning:")
print(df_clean.isna().sum())

df_clean.to_csv(os.path.join(table_dir, "credit_risk_clean.csv"), index=False)

## 4. Exploratory Data Analysis

In [ ]:
# Loan status distribution
default_counts = df_clean["loan_status"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(["Non-Default", "Default"], default_counts.values)
plt.title("Loan Status Distribution")
plt.xlabel("Loan Status")
plt.ylabel("Number of Borrowers")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "01_loan_status_distribution.png"), dpi=300)
plt.show()

print(default_counts)
print(df_clean["loan_status"].value_counts(normalize=True).sort_index())

In [ ]:
# Default rate by loan grade
grade_default = df_clean.groupby("loan_grade")["loan_status"].agg(
    borrower_count="count",
    default_count="sum",
    default_rate="mean"
).reset_index()

grade_default["default_rate"] = grade_default["default_rate"] * 100
grade_default.to_csv(os.path.join(table_dir, "default_rate_by_loan_grade.csv"), index=False)

plt.figure(figsize=(7, 4))
plt.bar(grade_default["loan_grade"], grade_default["default_rate"])
plt.title("Default Rate by Loan Grade")
plt.xlabel("Loan Grade")
plt.ylabel("Default Rate (%)")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "02_default_rate_by_loan_grade.png"), dpi=300)
plt.show()

grade_default

In [ ]:
# Default rate by home ownership
home_default = df_clean.groupby("person_home_ownership")["loan_status"].agg(
    borrower_count="count",
    default_count="sum",
    default_rate="mean"
).reset_index()

home_default["default_rate"] = home_default["default_rate"] * 100
home_default = home_default.sort_values("default_rate", ascending=False)
home_default.to_csv(os.path.join(table_dir, "default_rate_by_home_ownership.csv"), index=False)

plt.figure(figsize=(8, 4))
plt.bar(home_default["person_home_ownership"], home_default["default_rate"])
plt.title("Default Rate by Home Ownership")
plt.xlabel("Home Ownership")
plt.ylabel("Default Rate (%)")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "03_default_rate_by_home_ownership.png"), dpi=300)
plt.show()

home_default

In [ ]:
# Default rate by loan intent
intent_default = df_clean.groupby("loan_intent")["loan_status"].agg(
    borrower_count="count",
    default_count="sum",
    default_rate="mean"
).reset_index()

intent_default["default_rate"] = intent_default["default_rate"] * 100
intent_default = intent_default.sort_values("default_rate", ascending=False)
intent_default.to_csv(os.path.join(table_dir, "default_rate_by_loan_intent.csv"), index=False)

plt.figure(figsize=(9, 4))
plt.bar(intent_default["loan_intent"], intent_default["default_rate"])
plt.title("Default Rate by Loan Intent")
plt.xlabel("Loan Intent")
plt.ylabel("Default Rate (%)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "04_default_rate_by_loan_intent.png"), dpi=300)
plt.show()

intent_default

## 5. Feature Engineering

In [ ]:
model_df = df_clean.copy()

model_df["loan_to_income_ratio"] = model_df["loan_amnt"] / model_df["person_income"]
model_df["income_per_year_employed"] = model_df["person_income"] / (model_df["person_emp_length"] + 1)

model_df["high_loan_percent_income"] = np.where(
    model_df["loan_percent_income"] >= model_df["loan_percent_income"].quantile(0.75),
    1,
    0
)

model_df["high_interest_rate"] = np.where(
    model_df["loan_int_rate"] >= model_df["loan_int_rate"].quantile(0.75),
    1,
    0
)

target = "loan_status"

X = model_df.drop(columns=[target])
y = model_df[target]

X_encoded = pd.get_dummies(X, drop_first=True)

print("Feature data shape after encoding:", X_encoded.shape)
print("Target default rate:", y.mean())

X_encoded.to_csv(os.path.join(table_dir, "model_features_encoded.csv"), index=False)
y.to_csv(os.path.join(table_dir, "model_target.csv"), index=False)

## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Train default rate:", y_train.mean())
print("Test default rate:", y_test.mean())

## 7. Logistic Regression Baseline

In [ ]:
logit_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

logit_model.fit(X_train, y_train)

y_pred_logit = logit_model.predict(X_test)
y_prob_logit = logit_model.predict_proba(X_test)[:, 1]

logit_accuracy = accuracy_score(y_test, y_pred_logit)
logit_precision = precision_score(y_test, y_pred_logit)
logit_recall = recall_score(y_test, y_pred_logit)
logit_f1 = f1_score(y_test, y_pred_logit)
logit_auc = roc_auc_score(y_test, y_prob_logit)

print("===== Logistic Regression Performance =====")
print("Accuracy:", round(logit_accuracy, 4))
print("Precision:", round(logit_precision, 4))
print("Recall:", round(logit_recall, 4))
print("F1 Score:", round(logit_f1, 4))
print("AUC:", round(logit_auc, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_logit))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logit))

## 8. Random Forest Model

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=50,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_prob_rf)

print("===== Random Forest Performance =====")
print("Accuracy:", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall:", round(rf_recall, 4))
print("F1 Score:", round(rf_f1, 4))
print("AUC:", round(rf_auc, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
model_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [logit_accuracy, rf_accuracy],
    "Precision": [logit_precision, rf_precision],
    "Recall": [logit_recall, rf_recall],
    "F1_Score": [logit_f1, rf_f1],
    "AUC": [logit_auc, rf_auc]
})

model_results.to_csv(os.path.join(table_dir, "model_performance_summary.csv"), index=False)
model_results

In [ ]:
# ROC curve comparison
fpr_logit, tpr_logit, _ = roc_curve(y_test, y_prob_logit)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(7, 5))
plt.plot(fpr_logit, tpr_logit, label=f"Logistic Regression AUC = {logit_auc:.3f}")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest AUC = {rf_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Model")
plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "08_roc_curve_model_comparison.png"), dpi=300)
plt.show()

In [ ]:
# Random Forest feature importance
rf_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

rf_importance.to_csv(os.path.join(table_dir, "random_forest_feature_importance.csv"), index=False)

top_features = rf_importance.head(15).sort_values("Importance")

plt.figure(figsize=(8, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 15 Feature Importance - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "09_random_forest_feature_importance.png"), dpi=300)
plt.show()

rf_importance.head(15)

## 9. Risk Band Segmentation

In [ ]:
test_results = X_test.copy()
test_results["actual_default"] = y_test.values
test_results["predicted_pd"] = y_prob_rf
test_results["original_index"] = X_test.index

def assign_risk_band(pd_value):
    if pd_value < 0.05:
        return "Very Low Risk"
    elif pd_value < 0.10:
        return "Low Risk"
    elif pd_value < 0.20:
        return "Medium Risk"
    elif pd_value < 0.35:
        return "High Risk"
    else:
        return "Very High Risk"

test_results["risk_band"] = test_results["predicted_pd"].apply(assign_risk_band)

risk_band_summary = test_results.groupby("risk_band").agg(
    borrower_count=("actual_default", "count"),
    actual_default_count=("actual_default", "sum"),
    actual_default_rate=("actual_default", "mean"),
    avg_predicted_pd=("predicted_pd", "mean")
).reset_index()

risk_order = ["Very Low Risk", "Low Risk", "Medium Risk", "High Risk", "Very High Risk"]

risk_band_summary["risk_band"] = pd.Categorical(
    risk_band_summary["risk_band"],
    categories=risk_order,
    ordered=True
)

risk_band_summary = risk_band_summary.sort_values("risk_band")

risk_band_summary["actual_default_rate_pct"] = risk_band_summary["actual_default_rate"] * 100
risk_band_summary["avg_predicted_pd_pct"] = risk_band_summary["avg_predicted_pd"] * 100

risk_band_summary.to_csv(os.path.join(table_dir, "risk_band_summary.csv"), index=False)
risk_band_summary

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(risk_band_summary["risk_band"].astype(str), risk_band_summary["actual_default_rate_pct"])
plt.title("Actual Default Rate by Predicted Risk Band")
plt.xlabel("Predicted Risk Band")
plt.ylabel("Actual Default Rate (%)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "10_actual_default_rate_by_risk_band.png"), dpi=300)
plt.show()

## 10. Expected Loss Analysis

Expected Loss = PD × LGD × EAD

Assumptions:

- PD = predicted default probability
- LGD = 45%
- EAD = loan amount


In [ ]:
test_results["loan_amnt"] = df_clean.loc[test_results["original_index"], "loan_amnt"].values

LGD = 0.45

test_results["expected_loss"] = (
    test_results["predicted_pd"] * LGD * test_results["loan_amnt"]
)

expected_loss_summary = test_results.groupby("risk_band").agg(
    borrower_count=("actual_default", "count"),
    actual_default_count=("actual_default", "sum"),
    actual_default_rate=("actual_default", "mean"),
    avg_predicted_pd=("predicted_pd", "mean"),
    total_loan_amount=("loan_amnt", "sum"),
    avg_loan_amount=("loan_amnt", "mean"),
    total_expected_loss=("expected_loss", "sum"),
    avg_expected_loss=("expected_loss", "mean")
).reset_index()

expected_loss_summary["risk_band"] = pd.Categorical(
    expected_loss_summary["risk_band"],
    categories=risk_order,
    ordered=True
)

expected_loss_summary = expected_loss_summary.sort_values("risk_band")
expected_loss_summary["actual_default_rate_pct"] = expected_loss_summary["actual_default_rate"] * 100
expected_loss_summary["avg_predicted_pd_pct"] = expected_loss_summary["avg_predicted_pd"] * 100

expected_loss_summary.to_csv(os.path.join(table_dir, "expected_loss_by_risk_band.csv"), index=False)
expected_loss_summary

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(expected_loss_summary["risk_band"].astype(str), expected_loss_summary["total_expected_loss"])
plt.title("Total Expected Loss by Risk Band")
plt.xlabel("Predicted Risk Band")
plt.ylabel("Total Expected Loss")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "11_total_expected_loss_by_risk_band.png"), dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(expected_loss_summary["risk_band"].astype(str), expected_loss_summary["avg_expected_loss"])
plt.title("Average Expected Loss per Borrower by Risk Band")
plt.xlabel("Predicted Risk Band")
plt.ylabel("Average Expected Loss per Borrower")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "12_avg_expected_loss_by_risk_band.png"), dpi=300)
plt.show()

## 11. Approval Strategy Simulation

In [ ]:
def assign_decision(pd_value):
    if pd_value < 0.20:
        return "Approve"
    elif pd_value < 0.35:
        return "Manual Review"
    else:
        return "Reject"

test_results["decision"] = test_results["predicted_pd"].apply(assign_decision)

decision_summary = test_results.groupby("decision").agg(
    borrower_count=("actual_default", "count"),
    actual_default_count=("actual_default", "sum"),
    actual_default_rate=("actual_default", "mean"),
    total_loan_amount=("loan_amnt", "sum"),
    total_expected_loss=("expected_loss", "sum"),
    avg_expected_loss=("expected_loss", "mean")
).reset_index()

decision_order = ["Approve", "Manual Review", "Reject"]

decision_summary["decision"] = pd.Categorical(
    decision_summary["decision"],
    categories=decision_order,
    ordered=True
)

decision_summary = decision_summary.sort_values("decision")
decision_summary["actual_default_rate_pct"] = decision_summary["actual_default_rate"] * 100

decision_summary.to_csv(os.path.join(table_dir, "approval_strategy_summary.csv"), index=False)
decision_summary

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(decision_summary["decision"].astype(str), decision_summary["actual_default_rate_pct"])
plt.title("Actual Default Rate by Approval Decision")
plt.xlabel("Approval Decision")
plt.ylabel("Actual Default Rate (%)")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "13_default_rate_by_approval_decision.png"), dpi=300)
plt.show()

plt.figure(figsize=(7, 5))
plt.bar(decision_summary["decision"].astype(str), decision_summary["total_expected_loss"])
plt.title("Total Expected Loss by Approval Decision")
plt.xlabel("Approval Decision")
plt.ylabel("Total Expected Loss")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "14_expected_loss_by_approval_decision.png"), dpi=300)
plt.show()

In [ ]:
baseline_total_expected_loss = test_results["expected_loss"].sum()
baseline_borrowers = len(test_results)

approved_or_review = test_results[test_results["decision"] != "Reject"]

strategy_expected_loss = approved_or_review["expected_loss"].sum()
strategy_borrowers = len(approved_or_review)

loss_reduction = baseline_total_expected_loss - strategy_expected_loss
loss_reduction_pct = loss_reduction / baseline_total_expected_loss * 100

borrower_reduction = baseline_borrowers - strategy_borrowers
borrower_reduction_pct = borrower_reduction / baseline_borrowers * 100

strategy_impact = pd.DataFrame({
    "Metric": [
        "Baseline Expected Loss",
        "Strategy Expected Loss",
        "Expected Loss Reduction",
        "Expected Loss Reduction %",
        "Borrowers Rejected",
        "Borrowers Rejected %"
    ],
    "Value": [
        baseline_total_expected_loss,
        strategy_expected_loss,
        loss_reduction,
        loss_reduction_pct,
        borrower_reduction,
        borrower_reduction_pct
    ]
})

strategy_impact.to_csv(os.path.join(table_dir, "approval_strategy_impact.csv"), index=False)
strategy_impact

## 12. Business Conclusion

This project demonstrates how machine learning can support credit risk decision-making.

Key outcomes:

- Random Forest achieved stronger performance than Logistic Regression.
- Predicted default probabilities were translated into interpretable risk bands.
- Expected loss was calculated using a simplified PD × LGD × EAD framework.
- A risk-based approval strategy was simulated to reduce estimated expected loss.

This workflow connects predictive modeling with practical credit risk management.
